In [24]:
import  pandas as pd
import numpy as np

In [25]:
df=pd.read_csv('online_retail.csv')#导入数据

In [26]:
#查看数据
print('数据形状',df.shape)
print('数据的列')
print(df.columns.tolist())

数据形状 (541909, 8)
数据的列
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [27]:
#数据检查
print('数据类型\n',df.dtypes)

数据类型
 InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object


In [28]:
print('重复数据行数',df.duplicated().sum())

重复数据行数 5268


In [29]:
print('InvoiceNo去重个数',df['InvoiceNo'].nunique())

InvoiceNo去重个数 25900


In [30]:
print('Country去重个数',df['Country'].nunique())

Country去重个数 38


In [31]:
print('购买数量<0',(df['Quantity']<0).sum())

购买数量<0 10624


In [32]:
print('价格<0',(df['UnitPrice']<0).sum())

价格<0 2


In [33]:
print(df['Quantity'].describe())

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64


In [37]:
print(df['UnitPrice'].describe())

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


In [39]:
print(df.isna().sum(axis=0))

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [35]:
#清洗数据
clean_df=df.copy()

In [36]:
clean_df['InvoiceDate']=pd.to_datetime(clean_df['InvoiceDate'],errors='coerce')#将发票日期的dtype转变成日期类型

In [40]:
clean_df=clean_df.drop_duplicates()

In [41]:
clean_df=clean_df.dropna(subset=["CustomerID", "InvoiceDate"])

In [42]:
cancelled=clean_df['InvoiceNo'].astype(str).str.startswith(('C','c'))

In [43]:
print(cancelled.sum())#8872条取消订单

8872


In [44]:
clean_df=clean_df[~cancelled]#只取成功订单

In [45]:
clean_df=clean_df[(clean_df["Quantity"] > 0)&(clean_df["UnitPrice"] > 0)]

In [46]:
clean_df['TotalPrice']=(clean_df["Quantity"] * clean_df["UnitPrice"])

In [47]:
#确认数据

In [48]:
print("Cleaned shape:", clean_df.shape)
print("Customers:", clean_df["CustomerID"].nunique())
print("Invoices:", clean_df["InvoiceNo"].nunique())
print("Missing CustomerID:", clean_df["CustomerID"].isna().sum())
print("Negative Quantity:", (clean_df["Quantity"] <= 0).sum())
print("Negative UnitPrice:", (clean_df["UnitPrice"] <= 0).sum())

Cleaned shape: (392692, 9)
Customers: 4338
Invoices: 18532
Missing CustomerID: 0
Negative Quantity: 0
Negative UnitPrice: 0


In [50]:
#导出数据
clean_df.to_csv("clean_retail.csv", index=False, encoding="utf-8")
clean_df.to_excel("clean_retail.xlsx", index=False)

In [51]:
#制作RFM表
snapshop_data=clean_df["InvoiceDate"].max() + pd.Timedelta(days=1)
customer_table=clean_df.groupby('CustomerID').agg(
    Recency=(
        "InvoiceDate",
        lambda x:(snapshop_data-x.max()).days
    ),
    Frequency=('InvoiceNo','nunique'),
    Monetary=('TotalPrice','sum')
).reset_index()

In [52]:
customer_table.describe()

,CustomerID,Recency,Frequency,Monetary
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,92.536422,4.272015,2048.688081
std,1721.808492,100.014169,7.697998,8985.230220
min,12346.000000,1.000000,1.000000,3.750000
25%,13813.250000,18.000000,1.000000,306.482500
50%,15299.500000,51.000000,2.000000,668.570000
75%,16778.750000,142.000000,5.000000,1660.597500
max,18287.000000,374.000000,209.000000,280206.020000


In [53]:
customer_table.to_excel("customer_table.xlsx", index=False)